In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

import pandas as pd
# from ..src.convert_pangaea_textfile import align_cropland_columns
from src.alignment import align_cropland_columns

%load_ext autoreload
%autoreload 2

In [ ]:
df_cropland = pd.read_csv('../data/raw/N2o_ssa_cropland.tab', sep= '\t', encoding= 'utf-8')
df_forests = pd.read_csv('../data/raw/N2o_ssa_forests.tab', sep= '\t', encoding= 'utf-8')
df_grasslands = pd.read_csv('../data/raw/N2o_ssa_grasslands.tab', sep= '\t', encoding= 'utf-8')
df_africa = pd.read_csv('../data/processed/n2o_africa_clean.csv')

## Compare column structure

Before combining the datasets, the column names are compared across land-use types.  
This is necessary because the cropland file uses more explicit column names for some ERA5 soil variables.

In [ ]:
if 'Site info' in df_cropland.columns:
    print('df_cropland del "Site info"')
    df_cropland = df_cropland.drop(columns=['Site info'])
if 'Site info' in df_forests.columns:
    print('df_cropland del "Site info"')
    df_forests = df_forests.drop(columns=['Site info'])
if 'Site info' in df_grasslands.columns:
    print('df_grasslands del "Site info"')
    df_grasslands = df_grasslands.drop(columns=['Site info'])


print(f'df_cropland.columns {len(df_cropland.columns)}')
print(f'df_forests.columns {len(df_forests.columns)}')
print(f'df_grasslands.columns {len(df_grasslands.columns)}')
print(f'df_africa.columns {len(df_africa.columns)}')

df_cols = pd.DataFrame({
    'df_cropland' : df_cropland.columns,
    'df_forests' : df_forests.columns ,
    'df_grasslands' : df_grasslands.columns,
    'df_africa' : df_africa.columns
})
df_cols["cropland_diff_forests"] = df_cols["df_cropland"].ne(df_cols["df_forests"])
df_cols["cropland_diff_grasslands"] = df_cols["df_cropland"].ne(df_cols["df_grasslands"])
df_cols["forests_diff_grasslands"] = df_cols["df_forests"].ne(df_cols["df_grasslands"])

df_cols["africa_diff_forests"] = df_cols["df_africa"].ne(df_cols["df_forests"])
df_cols["africa_diff_cropland"] = df_cols["df_africa"].ne(df_cols["df_cropland"])
df_cols["africa_diff_grasslands"] = df_cols["df_africa"].ne(df_cols["df_grasslands"])

df_cols_diff = df_cols[df_cols.filter(like="_diff_").any(axis=1)]
df_cols_diff

# display(df_africa.iloc[:, 11:16])
# display(df_cropland.iloc[:, 11:16])
# display(df_forests.iloc[:, 11:16])
# display(df_grasslands.iloc[:, 11:16])

## Align cropland column names

The cropland dataset contains more detailed names for selected soil moisture and soil temperature columns.  
These columns are renamed to match the naming convention used in the forest and grassland datasets.

In [ ]:
df_cropland = align_cropland_columns(df_cropland)


## Combine land-use datasets

After aligning the column names, the three land-use datasets are concatenated row-wise into a single dataframe.

In [ ]:
df_cols = pd.DataFrame({
    'df_cropland' : df_cropland.columns,
    'df_forests' : df_forests.columns ,
    'df_grasslands' : df_grasslands.columns,
    'df_africa' : df_africa.columns
})
df_cols["cropland_diff_forests"] = df_cols["df_cropland"].ne(df_cols["df_forests"])
df_cols["cropland_diff_grasslands"] = df_cols["df_cropland"].ne(df_cols["df_grasslands"])
df_cols["forests_diff_grasslands"] = df_cols["df_forests"].ne(df_cols["df_grasslands"])

df_cols["africa_diff_forests"] = df_cols["df_africa"].ne(df_cols["df_forests"])
df_cols["africa_diff_cropland"] = df_cols["df_africa"].ne(df_cols["df_cropland"])
df_cols["africa_diff_grasslands"] = df_cols["df_africa"].ne(df_cols["df_grasslands"])

df_cols_diff = df_cols[df_cols.filter(like="_diff_").any(axis=1)]
df_cols_diff

## Save interim dataset

The combined dataset is saved as an interim file because only column alignment and concatenation have been applied.  
Further cleaning steps are performed later.

In [ ]:
df_all = pd.concat(
    [df_forests, df_cropland, df_grasslands],
    ignore_index=True,
    join="outer"
)
df_all.describe()

In [ ]:
out_dir = Path("../data/interim")
out_dir.mkdir(parents=True, exist_ok=True)

df_all.to_csv(
    out_dir / "n2o_ssa_landuse_aligned_combined.csv",
    index=False
)